# Algorithm Documentation

This notebook documents every algorithm implemented in this project, organised into three groups:

1. **Baseline algorithms** — standard KNN and its direct extensions
2. **Adaptive-k algorithms** — per-query k selection strategies (early exploration)
3. **FairRank family** — the main contribution: rank-correction for class imbalance

For each algorithm the document covers:
- What problem it solves
- How it works (mechanics and key equations)
- Design decisions and parameters
- **Empirical results**, for algorithms that appear in the final benchmark

Results come directly from `entrega_final.ipynb` (5-rep × 10-fold CV, 40 datasets) and `meta_analysis.ipynb` (LODO-CV instance space). Algorithms that did not make it into the final benchmark have no results section.

---

## 0. The Class-Imbalance Problem

Standard KNN votes by majority among the $k$ nearest neighbours. When the training set has far more majority-class points than minority-class points (imbalance ratio $r = N_{maj} / N_{min} \gg 1$), two things go wrong:

1. **Vote dilution.** Even near the true decision boundary, the $k$ neighbours are almost certainly dominated by majority points — just because there are more of them.
2. **Sampling bias in distances.** The expected minimum distance to the majority class is smaller than to the minority class, even when both classes have identical underlying densities. This is a pure statistical artefact from having more samples.

All algorithms in this project address one or both of these problems.

**Key tension that runs through all results:** G-Mean rewards algorithms that identify minority cases aggressively (high recall). MCC and F1 also penalise false positives, so they reward precision. An algorithm that predicts minority too eagerly wins G-Mean but loses MCC. This trade-off is the central thread of the empirical analysis.

**Final benchmark** (`entrega_final.ipynb §6`): 5-rep × 10-fold stratified CV across 40 datasets. Primary metric: G-Mean. Secondary: MCC, F1, ROC-AUC, PR-AUC. Statistical framework: Friedman + Wilcoxon with Holm correction (Demšar 2006).

---

## 1. Baseline Algorithms

### 1.1 KNNClassifier (Standard KNN)

**File:** `src/algorithms/baseline/knn_base.py`  
**Classes:** `KNNBase`, `KNNClassifier`, `KNNClassifierFast`

#### How it works

The standard $k$-Nearest Neighbours classifier:

1. Store all training points $\{(x_i, y_i)\}_{i=1}^N$.
2. For a query $x$, compute the distance $d(x, x_i)$ to every training point.
3. Take the $k$ points with the smallest distances — the $k$ nearest neighbours.
4. Return the most frequent class label among those $k$ neighbours.

$$\hat{y}(x) = \arg\max_{c} \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c]$$

$$P(y = c \mid x) = \frac{1}{k} \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c]$$

#### Class hierarchy

| Class | Description |
|---|---|
| `KNNBase` | Abstract base — stores data, sorts neighbours, delegates `aggregate()` |
| `KNNClassifier` | Implements `aggregate()` via `Counter.most_common(1)` |
| `KNNClassifierFast` | Same predictions; uses `scipy.cdist` for vectorised distance computation |

`KNNClassifierFast` replaces the Python loop with a single C-level matrix call — results are bit-for-bit identical, only speed differs. It is the base class for all other variants.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k` | 5 | Number of neighbours (`0` = use all) |
| `distance_func` | Euclidean | Any `scipy.spatial.distance` function |
| `n_jobs` | 1 | Parallelism via `joblib` |

#### Note on results

`KNNClassifier` with a fixed $k$ is not benchmarked in the final run — `KNNOptK` (below) is the standard-KNN representative, since selecting $k$ by CV is the correct baseline.

### 1.2 KNNOptK — KNN with Cross-Validated k

**File:** `src/algorithms/baseline/knn_base.py`  
**Class:** `KNNOptK`

#### How it works

Standard KNN requires choosing $k$ before seeing the data, which is a problem: a small $k$ is noisy, a large $k$ smears the boundary. KNNOptK turns this into a data-driven decision by running an inner cross-validation loop every time it trains.

**Step-by-step:**

1. From the training set of size $N$, build a candidate list of odd $k$ values:
   $$\{1, 3, 5, 7, \ldots, \lfloor\sqrt{N}\rfloor\}$$
   Odd values only — this avoids tie votes. The upper bound $\sqrt{N}$ scales automatically with dataset size (a dataset with 1600 samples considers up to $k=39$; one with 400 samples goes up to $k=19$).

2. Run a stratified $K$-fold split on the training data. For every combination of $(k, \text{fold})$, train KNNClassifierFast on the fold's training portion and evaluate balanced accuracy on the held-out portion. All $(k, \text{fold})$ pairs are evaluated in parallel.

3. Average the balanced accuracy across folds for each $k$ and pick the winner:
   $$k^* = \arg\max_k \; \overline{\text{BalAcc}}(k)$$

4. Train a final KNNClassifierFast with $k = k^*$ on the **full** training set.

**Why balanced accuracy, not plain accuracy?**  
With a 95/5 imbalance, a classifier that predicts majority on every query achieves 95% accuracy but 0% balanced accuracy. Balanced accuracy ($= (\text{TPR} + \text{TNR}) / 2$) is blind to class size, so the CV selects a $k$ that actually tries to identify minority cases.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_max` | `floor(sqrt(N))` | Upper bound on $k$ candidates |
| `inner_cv_folds` | from config | Folds for inner CV |
| `n_jobs` | 8 | Parallel $(k, \text{fold})$ evaluation |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§9`*

KNNOptK produces the most **counterintuitive result** in the entire benchmark. As stated in `entrega_final §11.1`:

> *KNNFairRankJointCV ranks #1 (mean G-Mean = 0.796 vs 0.631 for KNNOptK).*

| Metric | KNNOptK | Rank (out of 10) |
|---|---|---|
| G-Mean | 0.673 | **10th (last)** |
| MCC | 0.556 | 3rd |
| F1 | 0.586 | 4th |
| ROC-AUC | 0.803 | last |

It is **last on G-Mean** yet **3rd on MCC**. The instance space analysis (`entrega_final §9.2`) explains why:

> *KNNOptK wins MCC on 17/40 datasets (42%) despite ranking last on G-Mean. Under heavy class imbalance, predicting mostly majority yields a large TN that inflates MCC even when minority sensitivity is poor.*

The inner CV selects a small conservative $k$ that achieves high specificity but very low sensitivity — good MCC and F1, catastrophic G-Mean. Every FairRank variant beats KNNOptK on G-Mean with statistical significance ($p < 10^{-5}$, Holm-corrected).

### 1.3 KNNWeighted — Class-Frequency Weighting

**File:** `src/algorithms/baseline/knn_weighted.py`  
**Class:** `KNNWeighted`

#### How it works

The core idea is simple: minority votes count more. Instead of treating each neighbour's vote equally, multiply each class's vote count by a weight inversely proportional to its frequency.

The weight for class $c$ is:
$$w_c = \frac{N_{total}}{n_{classes} \cdot N_c}$$

**Concrete example** — suppose $N_{maj} = 950$, $N_{min} = 50$ (IR = 19):
$$w_{min} = \frac{1000}{2 \times 50} = 10, \quad w_{maj} = \frac{1000}{2 \times 950} \approx 0.53$$

Now suppose a query has $k = 5$ neighbours: 4 majority, 1 minority.  
- **Standard KNN:** majority wins 4–1. Predicts majority.  
- **KNNWeighted:** weighted score for majority = $4 \times 0.53 = 2.1$; for minority = $1 \times 10 = 10$. Predicts **minority**.

The normalised probability estimate is:
$$\tilde{P}(\text{minority} \mid x) = \frac{10}{10 + 2.1} \approx 0.83$$

This is what `predict_proba` returns — not the raw vote fraction, but the fraction of the total *weighted* vote.

#### Parameters

Inherits all parameters from `KNNClassifierFast`. The weights are computed at `fit` time from the training labels and stored as `_class_weights`.

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§9`*

KNNWeighted is the **sanity-check** baseline: if the main FairRank variants cannot beat it, the rank-correction mechanism adds nothing beyond frequency reweighting.

| Metric | KNNWeighted | Rank |
|---|---|---|
| G-Mean | 0.797 | 7th |
| MCC | 0.534 | 9th |
| F1 | 0.583 | 5th |
| ROC-AUC | 0.841 | 8th |

It beats KNNOptK clearly on G-Mean (+0.124) — confirming vote reweighting helps minority recall — but falls near the bottom on MCC, meaning the amplification overshoots and creates too many false positives.

The most revealing finding comes from the stress test (`entrega_final §9.3`):

> *KNNWeighted degrades as IR increases. It wins 20% of datasets in Q1 (mild imbalance) but 0% in Q3 and Q4.*

The fixed weight $w_{min} = r$ is a global constant. As $r$ grows, the weight grows with it — the algorithm over-corrects in majority-dense regions. In the **degenerate regime** (very few minority samples, `entrega_final §8`) the picture reverses: KNNWeighted ranks **1st** because it requires no inner CV and is immune to small-sample instability.

### 1.4 DANN — Discriminant Adaptive Nearest Neighbours

**File:** `src/algorithms/baseline/dann.py`  
**Class:** `DANN`  
**Reference:** Hastie & Tibshirani (1996)

#### How it works

Adapts the distance metric *locally* at each query point so that it stretches along discriminant directions and shrinks along within-class variation.

For each query $x$:
1. Find the 50 nearest Euclidean neighbours (local neighbourhood).
2. Compute within-class scatter $W = \sum_c \sum_{i \in \text{local}, y_i=c} (x_i-\mu_c)(x_i-\mu_c)^\top$.
3. Compute between-class scatter $B = \sum_c N_c (\mu_c - \mu)(\mu_c - \mu)^\top$.
4. Regularise: $W^* = W + \sigma I$.
5. Local metric: $M(x) = (W^*)^{-1} B (W^*)^{-1}$.
6. Adapted distance: $d_M(x, x_i) = \sqrt{(x-x_i)^\top M(x)(x-x_i)}$.
7. Classify by majority vote among $k$ nearest neighbours under $d_M$.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k` | 5 | Neighbours for the final vote |
| `sigma` | 1.0 | Regularisation |
| `metric` | `"euclidean"` | Base metric for the initial neighbourhood |

#### Note on results

DANN is not included in the final benchmark in `entrega_final.ipynb`. It was explored in earlier phases but dropped, primarily because:
1. A per-query $p \times p$ matrix inversion is too expensive for a 40-dataset, 5-rep benchmark.
2. The adapted metric helps with boundary geometry but does not correct the vote dilution problem — the final vote among $k$ neighbours still suffers from the same structural imbalance bias.

---

## 2. Adaptive-k Algorithms

These algorithms select $k$ per query point rather than fixing it globally. They represent an early design direction that was eventually superseded by the FairRank approach.

### 2.1 KNNAdaptiveEntropy

**File:** `src/algorithms/adaptive_k/knn_adaptive_entropy.py`  
**Class:** `KNNAdaptiveEntropy`

#### How it works

For each query $x$, select $k$ by maximising the Shannon entropy of the neighbourhood class distribution:

$$H(k) = -\sum_c p_c(k) \log_2 p_c(k)$$

**Intuition:** High entropy = balanced mix = near the decision boundary. Maximising $H(k)$ finds the $k$ at which the neighbourhood is most informationally rich about the local class structure.

Rather than evaluating all $k$ candidates, a halve/double hill-climb is used — $O(\log N_{train})$ evaluations per query:

```
k_start = floor(sqrt(N_train))   # odd
halve downward while H improves
double upward while H improves
```

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 1 | Minimum $k$ |
| `k_max` | `floor(sqrt(N))` | Maximum $k$ |
| `smoothing` | 1e-9 | Prevents $\log 0$ |

#### Note on results

Not in the final benchmark. Maximising entropy is not equivalent to correcting for imbalance: on a strongly imbalanced dataset, entropy is maximised with a very large $k$ that includes enough minority samples to raise the proportion — not through a principled distance correction. The FairRank approach addresses the root cause directly.

### 2.2 DANNAdaptive — DANN with Adaptive k

**File:** `src/algorithms/baseline/dann_adaptive.py`  
**Class:** `DANNAdaptive`

#### How it works

Combines DANN's locally adapted metric with per-query $k$ selection via the same halve/double hill-climb. Two strategies for the $k$ criterion:

- **`"entropy"`** — maximise Shannon entropy $H(k)$ (same as KNNAdaptiveEntropy)
- **`"eigen"`** — maximise effective dimensionality: number of PCA components explaining ≥95% of neighbourhood variance

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 1 | Minimum $k$ |
| `k_max` | `floor(sqrt(N))` | Maximum $k$ |
| `sigma` | 1.0 | DANN regularisation |
| `adaptation_strategy` | `"entropy"` | `"entropy"` or `"eigen"` |

#### Note on results

Not in the final benchmark. Inherits both DANN's cost (matrix inversion per query) and the entropy-maximisation limitation. Not a priority for the final comparison.

---

## 3. The FairRank Family

This is the main contribution of the project. All variants share the same theoretical foundation.

### 3.0 The Core Insight — Why Standard KNN Is Structurally Biased

Imagine two classes with **identical spatial density** — same shape, same spread — but very different sizes: $N_{maj} = 1000$, $N_{min} = 50$ (IR = 20).

Pick a query $x$ sitting right on the true decision boundary. Even though both classes are equally dense there, the nearest majority neighbour is almost certainly closer than the nearest minority neighbour. Why? Simply because with 1000 majority points there are more of them in every direction, and random sampling guarantees that some land very close to $x$ by chance.

**This is not a signal about the data. It is pure sampling noise from having more points.**

Formally, under a homogeneous Poisson process model the expected distance to the $k$-th nearest class-$c$ neighbour scales as:
$$E[d_k^c] \propto \left(\frac{k}{N_c}\right)^{1/d}$$

We want to find the majority rank $k_{eff}$ such that the $k_{eff}$-th majority neighbour is at the **same expected distance** as the 1st minority neighbour:
$$E[d_{k_{eff}}^{maj}] = E[d_1^{min}]$$
$$\left(\frac{k_{eff}}{N_{maj}}\right)^{1/d} = \left(\frac{1}{N_{min}}\right)^{1/d}$$

The dimension $d$ cancels on both sides:
$$\frac{k_{eff}}{N_{maj}} = \frac{1}{N_{min}} \implies \boxed{k_{eff} = \frac{N_{maj}}{N_{min}} = r}$$

**The fair rank is dimension-free** — it does not depend on how many features the data has.

**Intuition in plain language:** With IR = 20, you need to look at the 20th majority neighbour to find one that is at the same *statistically expected* distance as the 1st minority neighbour. The 1st majority neighbour is not a fair comparison — it wins by sheer numbers, not by proximity signal.

---

### 3.1 KNNFairRank (v3 — Core Algorithm)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank.py`  
**Class:** `KNNFairRank`

#### How it works

The algorithm replaces the single joint neighbourhood of standard KNN with **two separate per-class distance lists**, and then runs a series of *rank-corrected comparisons* to decide the class.

**At fit time:**
1. Split the training set: store $X_{min}$ (all minority points) and $X_{maj}$ (all majority points) separately.
2. Compute $r = N_{maj} / N_{min}$ and set $k_{eff} = \text{round}(r)$.
3. Size the majority neighbourhood generously — it needs to be large enough to cover $n_{votes} \times k_{eff}$ positions:
   $$k_{maj} = \max\left(\lceil r \rceil \cdot n_{votes} + \text{buffer},\; k_{floor}\right), \quad \text{capped at } k_{cap}$$

**At predict time — step by step for a query $x$:**

1. Compute distances from $x$ to every minority point, sort them ascending:
   $$d_1^{min} \leq d_2^{min} \leq \cdots \leq d_{k_{min}}^{min}$$

2. Do the same for the majority class:
   $$d_1^{maj} \leq d_2^{maj} \leq \cdots \leq d_{k_{maj}}^{maj}$$

3. Run $n_{votes}$ rank-corrected binary comparisons. For comparison $i$:
   - Take the $i$-th nearest **minority** point: $d_i^{min}$
   - Take the $(i \cdot k_{eff})$-th nearest **majority** point: $d_{i \cdot k_{eff}}^{maj}$
   - Vote minority if $d_i^{min} < d_{i \cdot k_{eff}}^{maj}$, else vote majority.

4. Predict **minority** if more than half the votes go minority; otherwise **majority**.

**Concrete example** — IR = 5, $k_{eff} = 5$, $n_{votes} = 3$:

| Comparison $i$ | Minority dist $d_i^{min}$ | Majority dist $d_{5i}^{maj}$ | Winner |
|:---:|:---:|:---:|:---:|
| 1 | 0.42 | 0.61 | minority ✓ |
| 2 | 0.55 | 0.50 | majority ✗ |
| 3 | 0.70 | 0.88 | minority ✓ |

Vote fraction = 2/3 > 0.5 → **predict minority**.

**Why this is fair:** Without the correction, the algorithm would compare $d_1^{min}$ to $d_1^{maj}$. Because of the sampling bias, $d_1^{maj}$ is almost always smaller — majority always wins. By comparing $d_1^{min}$ to $d_5^{maj}$ (the 5th majority neighbour), we reach the majority distance that is statistically equivalent to the 1st minority distance under equal densities. Only if the minority is genuinely closer than that does it win the vote.

**Why two separate lists, not one joint list?**  
With IR = 50 and $k = 5$, a joint neighbourhood almost never contains a minority point. Using separate lists guarantees both classes are always represented, regardless of imbalance.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 10 | Number of minority neighbours fetched |
| `k_maj_buffer` | 10 | Extra majority neighbours beyond the theoretical minimum |
| `k_maj_floor` | 30 | Minimum majority neighbourhood size (for stability at low IR) |
| `k_maj_cap` | 1000 | Cap on majority neighbourhood size (prevents memory blowup at extreme IR) |
| `n_votes` | 5 | Number of rank-corrected comparisons |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§9`*

KNNFairRank (v3) validates the core theory and exposes its main weakness:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.819 | 6th |
| MCC | 0.503 | **10th (last)** |
| ROC-AUC | 0.863 | 5th |

The G-Mean gap over KNNOptK is dramatic: 0.819 vs 0.673 on average, and at severe imbalance (Q1) 0.765 vs 0.565. The rank correction works — it correctly identifies minority cases that the standard vote would miss.

The low MCC reveals the weakness: the fixed $k_{eff} = r$ assumes uniform density everywhere. In majority-dense regions, $k_{eff} = r$ over-corrects — the algorithm predicts minority when it shouldn't, creating false positives that tank precision. The base v3 is the **anchor** of the FairRank family: maximum G-Mean correction, minimum MCC. Every subsequent variant tries to recover MCC by making the correction adaptive.

### 3.2 KNNFairRankMagnitude (Modification B)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_b.py`  
**Class:** `KNNFairRankMagnitude`

#### How it works

Replaces the binary vote $\mathbf{1}[d_i^{min} < d_{i \cdot k_{eff}}^{maj}]$ with a continuous confidence score:

$$s_i = \frac{d_{i \cdot k_{eff}}^{maj}}{d_i^{min} + d_{i \cdot k_{eff}}^{maj}} \in [0, 1]$$

- $s_i \to 1$: majority reference is much farther — strong minority signal
- $s_i = 0.5$: distances equal — no signal
- $s_i \to 0$: minority is much farther — strong majority signal

The mean $\bar{s}$ replaces the binary vote fraction. Hard decision: predict minority iff $\bar{s} > 0.5$.

#### Note on results

Not included in the final 10-algorithm benchmark in `entrega_final.ipynb`. Its main contribution — better-calibrated soft scores — is partly absorbed by KNNFairRankJackknife and KNNFairRankEnsemble, which also produce smoother probability estimates through averaging.

### 3.3 KNNFairRankCV (Modification C)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_c.py`  
**Class:** `KNNFairRankCV`

#### How it works

The problem with base v3 is that $k_{eff} = r$ is derived under the assumption that both classes have uniform density everywhere. When that assumption breaks down — minority clustered in sub-regions, majority spread unevenly — $k_{eff} = r$ over-corrects.

KNNFairRankCV introduces a single exponent $\alpha \in (0, 1]$ that dials the correction strength:

$$k_{eff} = \text{round}\left(r^\alpha\right)$$

**What $\alpha$ does:**

| $\alpha$ | $k_{eff}$ (for IR = 20) | Effect |
|:---:|:---:|---|
| 1.0 | 20 | Full v3 correction — maximum minority sensitivity |
| 0.75 | 9 | Moderate correction |
| 0.5 | $\approx$ 4 | Square-root damping — halfway between no correction and full |
| 0.25 | 2 | Mild correction — nearly no adjustment |

The key insight is that **$\alpha$ is not fixed** — it is chosen separately for each dataset by inner stratified cross-validation:

1. Split the training data into inner CV folds (default: 3-fold stratified).
2. For each candidate $\alpha \in \{0.25, 0.5, 0.75, 1.0\}$, train base KNNFairRank on the fold's training portion, temporarily set `clf._r = clf._r ** alpha`, and measure G-Mean (or another configured metric) on the held-out portion.
3. Average across folds and pick $\alpha^* = \arg\max_\alpha \overline{\text{G-Mean}}(\alpha)$.
4. Train the final KNNFairRank on the full training set and apply $\alpha^*$ at predict time.

**Why this fixes the over-correction problem:**  
On a dataset where both classes have uniform density, the CV will select $\alpha^* \approx 1$ (full correction). On a dataset where minority is clustered and the global $r$ overstates the local imbalance, the CV will select $\alpha^* < 1$, shrinking $k_{eff}$ and reducing false positives. The correction strength is now matched to the actual dataset structure.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Candidate correction exponents |
| `inner_cv_folds` | 3 | Folds for the inner CV |
| `scoring` | `geometric_mean` | Metric optimised to pick $\alpha$ |
| Plus all `KNNFairRank` parameters | — | — |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§7`*

KNNFairRankCV is one of the **best-balanced variants** in the final benchmark:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.826 | 2nd |
| MCC | 0.560 | 4th |
| F1 | 0.590 | 3rd |
| ROC-AUC | 0.867 | 3rd |

It is significantly better than base KNNFairRank on MCC and F1 (Wilcoxon, Holm-corrected), while maintaining equivalent G-Mean. The $\alpha$ tuning recovers precision without sacrificing recall.

The trade-off is the inner-CV fit overhead. On the degenerate subset (very few minority samples per fold, `entrega_final §8`), the CV becomes unreliable and KNNWeighted outperforms it.

### 3.4 KNNFairRankMagnitudeCV (Modification BC)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_bc.py`  
**Class:** `KNNFairRankMagnitudeCV`

#### How it works

Combines Modification B (continuous confidence scores) with Modification C (inner-CV $\alpha$ tuning). The inner CV uses magnitude-aware voting when evaluating $\alpha$ — using binary votes to select $\alpha$ then deploying with soft scores would be inconsistent.

$$k_{eff} = \text{round}(r^{\alpha^*}), \quad s_i = \frac{d_{i \cdot k_{eff}}^{maj}}{d_i^{min} + d_{i \cdot k_{eff}}^{maj}}$$

#### Note on results

Not included in the final benchmark in `entrega_final.ipynb`. The combination did not produce results significantly different from KNNFairRankCV alone — getting $\alpha$ right (Mod C) is the dominant improvement, and the continuous-score refinement (Mod B) is secondary. KNNFairRankJointCV (section 3.9), which tunes both $\alpha$ and $n_{votes}$ jointly, achieves better results than either modification in isolation.

### 3.5 KNNFairRankLocalOdds (Modification E)

**File:** `src/algorithms/fair_rank/local/knn_fair_rank_local_odds.py`  
**Class:** `KNNFairRankLocalOdds`

#### How it works

Replaces the global $k_{eff} = r$ with a **per-query** estimate of the local density ratio, using a counting trick that avoids dimensionality estimation.

For each minority rank $i$, count how many majority points are closer to $x$ than $d_i^{min}$:

$$j_i = \texttt{searchsorted}(d^{maj},\; d_i^{min})$$

Under homogeneous Poisson, $E[j_i / i] = r$. The median of these estimates is then Bayesian-shrunk toward the global $r$:

$$k_{eff}(x) = \text{round}\left(\frac{\text{median}_i(j_i/i) + \lambda r}{1 + \lambda}\right)$$

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_probe` | `k_min` | Minority ranks used to estimate local ratio |
| `shrinkage` | 1.0 | Prior weight $\lambda$ toward global $r$ |

#### Note on results

Not included in the final benchmark. The local estimation is theoretically sound but sensitive to the shrinkage hyperparameter $\lambda$: with $\lambda = 1$, the local estimate is heavily pulled toward the global $r$, nearly recovering v3; with small $\lambda$, the noisy per-point count $j_i/i$ dominates. Finding the right $\lambda$ per dataset requires its own CV, at which point KNNFairRankCV achieves the same goal with less complexity.

### 3.6 KNNFairRankEnsemble

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_ens.py`  
**Class:** `KNNFairRankEnsemble`

#### How it works

KNNFairRankCV picks one $(\alpha, n_{votes})$ setting via inner CV and commits to it. KNNFairRankEnsemble takes the opposite approach: **use the entire grid at prediction time** and pool all votes together.

For every combination $(n_v, \alpha) \in \{1,2,3,5,7,10\} \times \{0.25,0.5,0.75,1.0\}$ (24 pairs in total), run the binary rank-corrected vote step with $k_{eff} = \text{round}(r^\alpha)$ and $n_v$ comparisons. Pool every individual vote across all pairs:

$$\text{vote fraction} = \frac{\text{total minority wins across all 24 pairs}}{\text{total votes cast across all 24 pairs}}$$

Predict minority if this fraction $> 0.5$.

**Worked example** — IR = 10 (so $r = 10$), query $x$:

| $(n_v, \alpha)$ | $k_{eff}$ | Votes cast | Minority wins |
|:---:|:---:|:---:|:---:|
| (1, 0.25) | 2 | 1 | 0 |
| (1, 0.5) | 3 | 1 | 1 |
| (1, 1.0) | 10 | 1 | 1 |
| (3, 0.5) | 3 | 3 | 2 |
| (3, 1.0) | 10 | 3 | 2 |
| (5, 1.0) | 10 | 5 | 3 |
| … | … | … | … |
| **Total** | — | **e.g. 60** | **e.g. 33** |

Vote fraction = 33/60 = 0.55 → predict minority.

**Why pooling works better than picking one pair:**  
- Pairs with more votes ($n_v$ large) contribute proportionally more — naturally downweights the noisiest single-vote pairs.  
- The conservative $\alpha \in \{0.25, 0.5\}$ pairs use smaller $k_{eff}$ and thus vote minority less aggressively. They act as a brake on the full-correction pairs.  
- When all pairs agree (fraction near 0 or 1), the prediction is high-confidence. When they disagree (fraction near 0.5), the prediction is conservative — and the 0.5 threshold tends to classify these as majority, reducing false positives.  
- This diversity of correction strengths produces soft scores spread across $[0, 1]$ rather than quantised to $\{0, 1/n_v, \ldots, 1\}$, which directly benefits ROC-AUC.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Correction exponent candidates |
| `n_votes_grid` | [1, 2, 3, 5, 7, 10] | Vote count candidates |
| Plus `k_min`, `k_maj_*` from `KNNFairRank` | — | — |

#### Results and analysis

> *Source: `entrega_final.ipynb §6, §11.1`*

KNNFairRankEnsemble is the **ranking metric specialist** of the benchmark:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.772 | 9th |
| MCC | **0.582** | **1st** |
| F1 | **0.614** | **1st** |
| ROC-AUC | **0.884** | **1st** |
| PR-AUC | **0.649** | **1st** |

From `entrega_final §11.1`: it is **statistically significantly better than SMOTE+KNN** on ROC-AUC and PR-AUC. The G-Mean drops to 9th because the conservative $\alpha$ pairs reduce minority recall — borderline cases land near 0.5 and get classified as majority.

**When to prefer this algorithm:** when ROC-AUC or PR-AUC is the primary metric, or when precision-recall balance matters more than maximising minority recall.

### 3.7 KNNFairRankTopoJoint

**File:** `src/algorithms/fair_rank/topology/knn_fair_rank_topo_joint.py`  
**Class:** `KNNFairRankTopoJoint`

#### How it works

Partitions the training space into regions via Ward linkage on the joint point cloud, then assigns a per-region $k_{eff}$ based on the local imbalance ratio.

**At fit time:**
1. Build a Ward dendrogram on all $N$ training points.
2. Find the cut with the largest relative gap (gap / cloud diameter) where every region has $\geq$ `min_region_samples` points. Fall back to one global region if no valid cut exists.
3. For each region $R_j$: $k_{eff}(R_j) = \min\left(\frac{N_{maj}(R_j) + \lambda}{N_{min}(R_j) + \lambda},\ r\right)$

**At predict time:** assign $x$ to the region of its nearest training point, use that region's $k_{eff}$.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `min_persistence_ratio` | 0.05 | Minimum gap/diameter for a split |
| `laplace_smooth` | 1.0 | $\lambda$ in per-region $k_{eff}$ |
| `min_region_samples` | adaptive | Min points per region |

#### Results and analysis

> *Source: `entrega_final.ipynb §6`*

KNNFairRankTopoJoint achieves **consistently mid-tier performance** across all metrics:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.821 | 5th |
| MCC | 0.543 | 7th |
| ROC-AUC | 0.862 | 6th |

It never leads any metric, nor is it last. This reflects the nature of the topology approach: the Ward clustering finds meaningful spatial partitions when the minority class genuinely occupies distinct sub-regions of the feature space. On those datasets, the per-region $k_{eff}$ is more appropriate than the global $r$, and performance improves. On datasets where minority points are uniformly distributed throughout the space, the algorithm falls back to a single global region, recovering base v3. Across a diverse 40-dataset benchmark these two behaviours average out to a middle ranking.

The unique advantage of this variant is **interpretability**: the `zone_counts_` attribute classifies training regions into majority-dominated, minority-dominated, and mixed zones, and `point_k_eff_` gives a per-training-point correction strength — useful for understanding where in the feature space the imbalance is most severe.

### 3.8 KNNFairRankJackknife

**File:** `src/algorithms/fair_rank/resampling/knn_fair_rank_jackknife.py`  
**Class:** `KNNFairRankJackknife`

#### How it works

Base v3 has a fragility: if a single minority training point happens to land very close to a query $x$ — due to a duplicate, a mislabelled sample, or random sampling noise — it wins comparison $i=1$ by a large margin and can tip the vote toward minority regardless of what the other comparisons say.

KNNFairRankJackknife makes the decision robust to this by running **Leave-One-Out trials over the minority distance sequence**.

**Step by step for a query $x$:**

1. Fetch sorted minority distances $[d_1^{min}, d_2^{min}, \ldots, d_{k_{min}}^{min}]$ and majority distances as usual.

2. For each trial $j = 1, \ldots, k_{probe}$:
   - **Remove** $d_j^{min}$ from the minority list — pretend that training point does not exist.
   - Run the standard v3 vote step on the remaining $k_{min} - 1$ distances (ranks shift down by one after the removal).
   - Record the resulting vote fraction $f_j$.

3. Average across all trials:
   $$\bar{f} = \frac{1}{k_{probe}} \sum_{j=1}^{k_{probe}} f_j$$

4. Predict minority if $\bar{f} > 0.5$.

**Concrete illustration** — $n_{votes} = 3$, $k_{probe} = 3$:

Suppose a noisy minority point at rank 1 is unusually close. Without jackknife, $d_1^{min}$ easily wins its comparison, driving the vote fraction high. With jackknife:

| Trial $j$ | Point removed | Vote fraction $f_j$ |
|:---:|:---:|:---:|
| 1 | $d_1^{min}$ (the noisy one) | 0.33 |
| 2 | $d_2^{min}$ | 0.67 |
| 3 | $d_3^{min}$ | 0.67 |

$\bar{f} = (0.33 + 0.67 + 0.67)/3 = 0.56$ — still predicts minority, but the confidence is lower and more honest. If the noisy point had caused $f = 1.0$ in v3, the jackknife average tempers it to 0.56.

If removing a point changes the vote fraction substantially, the average is pulled toward the centre — the prediction is more conservative and less susceptible to the anomaly. If no removal changes anything, $\bar{f} = f_{v3}$ exactly and there is no cost.

**Constraint on `k_probe`:**  
After removing one minority point, at least $n_{votes}$ distances must remain for the vote step. So `k_probe` is automatically capped at `k_min - n_votes`. If too few minority points are available (degenerate datasets), the jackknife falls back to standard v3.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_probe` | `n_votes` | Number of LOO trials (auto-capped at `k_min - n_votes`) |
| Plus all `KNNFairRank` parameters | — | — |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§7`*

KNNFairRankJackknife is one of the most **consistently strong** variants:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.819 | 6th |
| MCC | 0.530 | 6th |
| ROC-AUC | **0.873** | **2nd** |

The Wilcoxon tests show it produces some of the strongest MCC improvements over base KNNFairRank — the LOO averaging eliminates the systematic misclassification from anomalous minority points. The strong ROC-AUC (2nd) comes from the same source: jackknife vote fractions are better-calibrated soft scores because they reflect genuine uncertainty rather than false confidence from a single lucky neighbour. No inner CV is needed at fit time, so it remains reliable on small or degenerate datasets.

### 3.9 KNNFairRankJointCV

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_joint_cv.py`  
**Class:** `KNNFairRankJointCV`

#### How it works

KNNFairRankCV (Mod C) tunes $\alpha$ by inner CV while keeping $n_{votes}$ fixed at its default. But $\alpha$ and $n_{votes}$ are not independent parameters — they interact:

- High $n_{votes}$ with low $\alpha$: many conservative comparisons. The algorithm sees a lot of evidence, but each comparison is less corrected.
- Low $n_{votes}$ with high $\alpha$: few aggressive comparisons. Less evidence, but each comparison applies the full correction.

Optimising one while fixing the other finds a local optimum. KNNFairRankJointCV searches the **full grid jointly**:

$$\text{Grid: } (n_{votes}, \alpha) \in \{1, 2, 3, 5, 7, 10\} \times \{0.25, 0.5, 0.75, 1.0\}$$

**The inner CV procedure:**

1. For every pair $(n_v, \alpha)$ in the grid (24 combinations), run stratified $K$-fold CV on the training data.
2. For each fold, train KNNFairRank, set `clf._r = clf._r ** alpha` and `clf._n_votes = n_v`, and score on the held-out fold using G-Mean (configurable).
3. Average across folds for each pair. Pick the winner:
   $$(n_v^*, \alpha^*) = \arg\max_{(n_v, \alpha)} \overline{\text{G-Mean}}(n_v, \alpha)$$
4. Train the final KNNFairRank with $n_{votes} = n_v^*$ and apply $\alpha^*$ at predict time.

**Why this beats separate tuning:**  
Suppose $\alpha = 1.0$ wins when tested with the default $n_{votes} = 5$. But with $n_{votes} = 3$, maybe $\alpha = 0.75$ scores higher because a tighter correction over fewer votes better matches that dataset's density. The joint search discovers these interactions; Mod C alone cannot.

The fit-time cost is the highest of all variants — 24 pairs × folds evaluations — but the performance payoff is the best overall balance in the benchmark.

#### Results and analysis

> *Source: `entrega_final.ipynb §6, §9, §11–§12` and `meta_analysis.ipynb §6–§7`*

KNNFairRankJointCV is the **overall winner** — best G-Mean, second best MCC:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | **0.828** | **1st** |
| MCC | 0.574 | 2nd |
| F1 | 0.606 | 2nd |
| ROC-AUC | 0.863 | 4th |

From `entrega_final §11.1`:

> *KNNFairRankJointCV ranks #1 (mean G-Mean = 0.796 vs 0.631 for KNNOptK). It is not significantly better than SMOTE+KNN after Holm correction (p_corr = 0.136, \u0394 = +0.030), but leads the ranking consistently.*

The instance space analysis (`entrega_final §9.3`) gives the strongest validation result:

> *KNNFairRankJointCV is the only algorithm whose G-Mean footprint grows from Q1 (20%) to Q4 (30%). Its relative advantage increases as imbalance becomes more severe.*

The meta-analysis notebook (`meta_analysis.ipynb`) confirms this predictively: a Leave-One-Dataset-Out meta-classifier trained on dataset meta-features can predict above chance whether KNNFairRankJointCV will beat KNNOptK, and the win rate increases monotonically across IR quartiles.

**Limitation in the degenerate regime:** With very few minority samples per fold, the inner CV over 24 pairs becomes unreliable and KNNFairRankJointCV drops to 4th. KNNWeighted, which requires no optimisation, outperforms it in this edge case.

---

## 4. Overall Comparison

### Mean scores and ranks (5-rep × 10-fold CV, 40 datasets)

Sorted by G-Mean rank:

| Algorithm | G-Mean | MCC | F1 | ROC-AUC | G-Mean rank | MCC rank |
|---|---|---|---|---|---|---|
| **KNNFairRankJointCV** | **0.828** | 0.574 | 0.606 | 0.863 | **1** | 2 |
| KNNFairRankCV | 0.826 | 0.560 | 0.590 | 0.867 | 2 | 4 |
| KNNFairRankOptVotes | 0.828 | 0.537 | 0.568 | 0.857 | 3 | 8 |
| SMOTE+KNN | 0.807 | 0.550 | 0.595 | 0.850 | 4 | 5 |
| KNNFairRankTopoJointBootstrap | 0.821 | 0.543 | 0.577 | 0.862 | 5 | 7 |
| KNNFairRankJackknife | 0.819 | 0.530 | 0.559 | 0.873 | 6 | 6 |
| KNNWeighted | 0.797 | 0.534 | 0.583 | 0.841 | 7 | 9 |
| KNNFairRank | 0.819 | 0.503 | 0.534 | 0.863 | 7 | 10 |
| **KNNFairRankEnsemble** | 0.772 | **0.582** | **0.614** | **0.884** | 9 | **1** |
| KNNOptK | 0.673 | 0.556 | 0.586 | 0.803 | 10 | 3 |

### The central trade-off (`entrega_final §11.1`)

> *"No single algorithm dominates all metrics — the results split cleanly by metric type."*

- **Best for G-Mean** (maximise minority recall): `KNNFairRankJointCV`
- **Best for MCC, F1, ROC-AUC, PR-AUC** (balance precision and recall): `KNNFairRankEnsemble`
- **Surprising MCC performer**: `KNNOptK` ranks 3rd on MCC despite being last on G-Mean — a conservative baseline that achieves high specificity by rarely predicting minority
- **Degenerate datasets** (very few minority samples): `KNNWeighted` — no inner CV dependency

### MCC pathology (`entrega_final §9.2`)

> *"KNNOptK wins MCC on 17/40 datasets (42%) despite ranking last on G-Mean. This pattern confirms that MCC alone is an unreliable criterion for imbalanced classification and should not be used as the sole evaluation metric."*

### G-Mean grows with imbalance severity (`entrega_final §9.3`)

KNNFairRankJointCV is the **only algorithm whose G-Mean win rate increases from mild (Q1: 20%) to severe imbalance (Q4: 30%)**. All other algorithms either stay flat or degrade. This confirms the theoretical prediction: the fair-rank correction $k_{eff} = r^\alpha$ becomes more important — and more accurate — as the imbalance ratio grows.

---

*This document was generated from source code in `src/algorithms/`. Results and interpretations are sourced directly from `notebooks/entrega_final.ipynb` and `notebooks/meta_analysis.ipynb`.*